# Adversarial Robustness of Machine Learning-Based Intrusion Detection Systems

This notebook combines the strongest parts of our two project notebooks:

- **Amogh's notebook:** multiclass CICIDS2017 intrusion detection, PyTorch MLP model, FGSM attack, PGD attack, and transferability testing across Logistic Regression and Random Forest.
- **Our preprocessing notebook:** stronger data-cleaning habits, stratified train/test split, confusion matrices, classification reports, artifact saving, and a more reproducible workflow.

The goal is to create a cleaner, GitHub-ready notebook that can serve as the main experimental notebook for the project.


## 1. Project Goal

Machine-learning intrusion detection systems can perform very well on clean benchmark traffic, but they may be vulnerable to adversarial perturbations. In this experiment, we:

1. Load and clean CICIDS2017 network-flow data.
2. Train baseline intrusion-detection models.
3. Train a neural-network IDS model.
4. Evaluate all models on clean test data.
5. Generate adversarial test samples using FGSM and PGD against the neural network.
6. Test whether the adversarial examples transfer to non-neural models.

The default path uses the cleaned/preprocessed CICIDS2017 dataset with the multiclass label column **`Attack Type`**. A raw-CSV fallback is included for local CICIDS2017 files that use the original **`Label`** column.


## 2. Setup and Configuration

Edit the paths in this section only if the notebook cannot automatically find the dataset.


In [ ]:
import json
import os
import random
import warnings
from pathlib import Path

import glob
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------------
# Reproducibility
# -----------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# -----------------------------
# Data paths
# -----------------------------
# The cleaned multiclass dataset used in Amogh's Kaggle notebook.
CLEANED_CSV_CANDIDATES = [
    Path("data/cicids2017_cleaned.csv"),
    Path("../data/cicids2017_cleaned.csv"),
    Path("/kaggle/input/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/cicids2017_cleaned.csv"),
]

# Raw CICIDS2017 CSV fallback, based on the preprocessing notebook.
RAW_CSV_GLOBS = [
    "data/raw/*.csv",
    "../data/raw/*.csv",
    "*.csv",
]

# -----------------------------
# Experiment settings
# -----------------------------
TEST_SIZE = 0.20
BATCH_SIZE = 1024
MLP_EPOCHS = 5
MLP_LEARNING_RATE = 1e-3
DROPOUT = 0.30

# To keep classical baselines fast, train Logistic Regression and Random Forest
# on a stratified subset when the training data is very large.
SKLEARN_TRAIN_LIMIT = 300_000

# Adversarial attack settings from Amogh's first pass.
FGSM_EPSILON = 0.05
PGD_EPSILON = 0.05
PGD_ALPHA = 0.01
PGD_STEPS = 10

OUTPUT_DIR = Path("outputs")
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 3. Data Loading and Cleaning Helpers

This section keeps the notebook flexible:

- If the cleaned Kaggle-style file exists, the notebook uses the multiclass **`Attack Type`** label.
- If that file is not found, the notebook looks for raw CICIDS2017 CSV files and converts the original **`Label`** column into a binary target: `BENIGN = 0`, attack traffic = `1`.

The raw fallback incorporates the stronger preprocessing pattern from our preprocessing notebook: strip column names, remove duplicate columns/rows, replace infinities, drop all-null/constant columns, coerce features to numeric, and median-impute missing values.


In [ ]:
def first_existing_path(candidates):
    """Return the first existing file path from a list of candidate paths."""
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None


def load_cleaned_cicids2017(csv_path):
    """Load the cleaned/preprocessed multiclass CICIDS2017 dataset."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.replace([np.inf, -np.inf], np.nan)

    label_col = "Attack Type"
    if label_col not in df.columns:
        raise ValueError(f"Expected '{label_col}' column in cleaned dataset, but it was not found.")

    y = df[label_col].astype(str)
    X = df.drop(columns=[label_col])
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))

    # Drop any columns that are still entirely missing after numeric coercion.
    all_null_cols = X.columns[X.isna().all()].tolist()
    if all_null_cols:
        X = X.drop(columns=all_null_cols)

    # Fill any remaining missing values with zero as a final guardrail.
    X = X.fillna(0)

    metadata = {
        "source": "cleaned_multiclass",
        "path": str(csv_path),
        "target_name": label_col,
        "task": "multiclass",
        "description": "Cleaned/preprocessed CICIDS2017 dataset with Attack Type labels.",
    }
    return X, y, metadata


def load_raw_cicids2017(csv_files):
    """Load raw CICIDS2017 CSV files and prepare a binary BENIGN-vs-attack target."""
    if not csv_files:
        raise FileNotFoundError("No raw CICIDS2017 CSV files were found.")

    df_list = []
    for file in csv_files:
        temp = pd.read_csv(file)
        temp["__source_file"] = Path(file).name
        df_list.append(temp)

    df = pd.concat(df_list, ignore_index=True)
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.drop_duplicates()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=1, how="all")

    label_col = "Label"
    if label_col not in df.columns:
        raise ValueError(f"Expected raw CICIDS2017 column '{label_col}', but it was not found.")

    constant_cols = [col for col in df.columns if col != label_col and df[col].nunique(dropna=False) <= 1]
    if constant_cols:
        df = df.drop(columns=constant_cols)

    X = df.drop(columns=[label_col])
    X = X.drop(columns=["__source_file"], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    X = X.fillna(0)

    # Binary target: 0 = BENIGN, 1 = any attack.
    y = df[label_col].astype(str).str.strip().apply(lambda label: "BENIGN" if label.upper() == "BENIGN" else "ATTACK")

    metadata = {
        "source": "raw_binary",
        "path": [str(path) for path in csv_files],
        "target_name": label_col,
        "task": "binary",
        "description": "Raw CICIDS2017 CSV files converted to BENIGN-vs-ATTACK classification.",
    }
    return X, y, metadata


def find_raw_csv_files(patterns):
    """Find raw CSV files from a list of glob patterns, excluding known outputs."""
    files = []
    exclude_names = {"results_summary.csv"}
    for pattern in patterns:
        files.extend(glob.glob(pattern))
    files = sorted({Path(file) for file in files if Path(file).name not in exclude_names})
    return files


def load_dataset():
    """Auto-detect and load either the cleaned multiclass dataset or raw CICIDS2017 CSVs."""
    cleaned_path = first_existing_path(CLEANED_CSV_CANDIDATES)
    if cleaned_path is not None:
        print(f"Loading cleaned multiclass dataset from: {cleaned_path}")
        return load_cleaned_cicids2017(cleaned_path)

    raw_files = find_raw_csv_files(RAW_CSV_GLOBS)
    if raw_files:
        print("Cleaned dataset not found. Loading raw CSV fallback files:")
        for file in raw_files[:10]:
            print(f"  - {file}")
        if len(raw_files) > 10:
            print(f"  ... and {len(raw_files) - 10} more")
        return load_raw_cicids2017(raw_files)

    searched = [str(path) for path in CLEANED_CSV_CANDIDATES] + RAW_CSV_GLOBS
    raise FileNotFoundError(
        "No CICIDS2017 data files were found. Place the cleaned file at data/cicids2017_cleaned.csv "
        "or place raw CICIDS2017 CSV files under data/raw/.\n\nSearched:\n- " + "\n- ".join(searched)
    )


## 4. Load the Dataset

The class distribution matters a lot for this project. CICIDS2017 is imbalanced, so we track more than just accuracy later in the notebook.


In [ ]:
X, y, data_metadata = load_dataset()
feature_names = X.columns.tolist()

print("Dataset source:", data_metadata["source"])
print("Task:", data_metadata["task"])
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

class_counts = y.value_counts().rename_axis("class_label").reset_index(name="count")
class_counts["percentage"] = 100 * class_counts["count"] / class_counts["count"].sum()
display(class_counts)

display(X.head())


## 5. Quick Class-Balance Check

Weighted F1 can look very strong on imbalanced data because large classes dominate the metric. We therefore also report macro F1 and balanced accuracy.


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(class_counts["class_label"].astype(str), class_counts["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Class Distribution")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()


## 6. Encode Labels, Split Data, and Scale Features

We use a stratified split so the train and test sets preserve the same class proportions. The `StandardScaler` is fit only on the training set to avoid test-data leakage.


In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y.astype(str))
class_names = label_encoder.classes_.tolist()

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_test_scaled = scaler.transform(X_test_df)

print("Classes:")
for index, name in enumerate(class_names):
    print(f"  {index}: {name}")

print("\nTrain shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

train_distribution = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_distribution = pd.Series(y_test).value_counts(normalize=True).sort_index()
split_check = pd.DataFrame({
    "class_name": class_names,
    "train_percentage": train_distribution.values * 100,
    "test_percentage": test_distribution.values * 100,
})
display(split_check)


## 7. Evaluation Helpers

These helpers keep the reporting consistent across models and attack settings.


In [ ]:
results = []


def compute_metrics(y_true, y_pred):
    """Return the main metrics used in the project."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def add_result(model_name, condition, y_true, y_pred, notes=""):
    """Add one model/condition result to the global results list."""
    metrics = compute_metrics(y_true, y_pred)
    row = {
        "model": model_name,
        "condition": condition,
        **metrics,
        "notes": notes,
    }
    results.append(row)
    return row


def show_classification_summary(model_name, condition, y_true, y_pred):
    """Print metrics and a classification report."""
    metrics = compute_metrics(y_true, y_pred)
    print(f"{model_name} - {condition}")
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


def plot_confusion(y_true, y_pred, title):
    """Plot a confusion matrix."""
    labels = np.arange(len(class_names))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    display_obj = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(9, 7))
    display_obj.plot(ax=ax, xticks_rotation=45, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def make_stratified_subset(X_array, y_array, max_samples, random_state=RANDOM_STATE):
    """Return a stratified subset for faster classical model training."""
    if max_samples is None or len(y_array) <= max_samples:
        return X_array, y_array

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=max_samples,
        random_state=random_state,
    )
    subset_indices, _ = next(splitter.split(X_array, y_array))
    return X_array[subset_indices], y_array[subset_indices]


## 8. Train Classical Baseline Models

We keep Logistic Regression and Random Forest because they make the project stronger: we can compare a neural model against non-neural baselines and later test whether adversarial examples transfer across model families.

For consistency with the adversarial samples, both classical baselines are trained on the scaled feature space.


In [ ]:
X_train_sklearn, y_train_sklearn = make_stratified_subset(
    X_train_scaled,
    y_train,
    max_samples=SKLEARN_TRAIN_LIMIT,
)

print("Classical model training subset:", X_train_sklearn.shape)

log_reg = LogisticRegression(
    max_iter=500,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
log_reg.fit(X_train_sklearn, y_train_sklearn)

random_forest = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_forest.fit(X_train_sklearn, y_train_sklearn)

sklearn_models = {
    "Logistic Regression": log_reg,
    "Random Forest": random_forest,
}

for model_name, model in sklearn_models.items():
    clean_preds = model.predict(X_test_scaled)
    add_result(model_name, "clean", y_test, clean_preds)
    show_classification_summary(model_name, "clean", y_test, clean_preds)
    plot_confusion(y_test, clean_preds, f"{model_name} - Clean Test Data")


## 9. Train the Neural Network IDS Model

This section keeps Amogh's PyTorch MLP idea, because it gives us a differentiable model for FGSM and PGD attacks.

Architecture:

- Input: one neuron per network-flow feature
- Hidden layer: 128 neurons + ReLU + dropout
- Hidden layer: 64 neurons + ReLU
- Output: one neuron per target class


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=DROPOUT):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.model(x)


X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

mlp = MLP(input_dim=X_train_scaled.shape[1], num_classes=len(class_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=MLP_LEARNING_RATE)

print(mlp)


In [ ]:
for epoch in range(MLP_EPOCHS):
    mlp.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        outputs = mlp(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{MLP_EPOCHS} - loss: {average_loss:.4f}")


In [ ]:
def predict_mlp(model, data_loader):
    """Return predictions and labels for a PyTorch model/data loader."""
    model.eval()
    predictions = []
    labels = []

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(DEVICE)
            outputs = model(batch_X)
            batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(batch_preds)
            labels.extend(batch_y.numpy())

    return np.array(labels), np.array(predictions)


mlp_clean_labels, mlp_clean_preds = predict_mlp(mlp, test_loader)
add_result("MLP", "clean", mlp_clean_labels, mlp_clean_preds)
show_classification_summary("MLP", "clean", mlp_clean_labels, mlp_clean_preds)
plot_confusion(mlp_clean_labels, mlp_clean_preds, "MLP - Clean Test Data")


## 10. Generate FGSM and PGD Adversarial Examples

FGSM and PGD perturb the scaled tabular feature vectors in the direction that increases the neural network's loss.

Important limitation: these attacks show model sensitivity in feature space, but they do not guarantee that every perturbed vector corresponds to a physically valid network flow. We should mention this limitation in the final report.


In [ ]:
feature_min = torch.tensor(X_train_scaled.min(axis=0), dtype=torch.float32, device=DEVICE)
feature_max = torch.tensor(X_train_scaled.max(axis=0), dtype=torch.float32, device=DEVICE)


def fgsm_attack(model, X_batch, y_batch, epsilon):
    """Fast Gradient Sign Method attack on scaled feature vectors."""
    model.eval()
    X_adv = X_batch.clone().detach().to(DEVICE)
    y_batch = y_batch.to(DEVICE)
    X_adv.requires_grad = True

    outputs = model(X_adv)
    loss = criterion(outputs, y_batch)

    model.zero_grad()
    loss.backward()

    perturbation = epsilon * X_adv.grad.sign()
    X_adv = X_adv + perturbation
    X_adv = torch.max(torch.min(X_adv, feature_max), feature_min)
    return X_adv.detach()


def pgd_attack(model, X_batch, y_batch, epsilon, alpha, steps):
    """Projected Gradient Descent attack on scaled feature vectors."""
    model.eval()
    X_original = X_batch.clone().detach().to(DEVICE)
    y_batch = y_batch.to(DEVICE)
    X_adv = X_original.clone().detach()

    for _ in range(steps):
        X_adv.requires_grad = True
        outputs = model(X_adv)
        loss = criterion(outputs, y_batch)

        model.zero_grad()
        loss.backward()

        X_adv = X_adv + alpha * X_adv.grad.sign()
        perturbation = torch.clamp(X_adv - X_original, min=-epsilon, max=epsilon)
        X_adv = X_original + perturbation
        X_adv = torch.max(torch.min(X_adv, feature_max), feature_min).detach()

    return X_adv


def generate_adversarial_dataset(model, data_loader, attack_name, **attack_kwargs):
    """Generate adversarial examples and evaluate the MLP on them."""
    model.eval()
    adv_batches = []
    label_batches = []
    predictions = []
    labels = []

    for batch_X, batch_y in data_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        if attack_name == "FGSM":
            adv_X = fgsm_attack(model, batch_X, batch_y, **attack_kwargs)
        elif attack_name == "PGD":
            adv_X = pgd_attack(model, batch_X, batch_y, **attack_kwargs)
        else:
            raise ValueError(f"Unknown attack: {attack_name}")

        adv_batches.append(adv_X.cpu())
        label_batches.append(batch_y.cpu())

        with torch.no_grad():
            outputs = model(adv_X)
            batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()

        predictions.extend(batch_preds)
        labels.extend(batch_y.cpu().numpy())

    X_adv = torch.cat(adv_batches).numpy()
    y_adv = torch.cat(label_batches).numpy()
    return X_adv, y_adv, np.array(labels), np.array(predictions)


In [ ]:
X_test_fgsm, y_test_fgsm, fgsm_labels, fgsm_preds = generate_adversarial_dataset(
    mlp,
    test_loader,
    attack_name="FGSM",
    epsilon=FGSM_EPSILON,
)

add_result("MLP", "FGSM", fgsm_labels, fgsm_preds, notes=f"epsilon={FGSM_EPSILON}")
show_classification_summary("MLP", f"FGSM epsilon={FGSM_EPSILON}", fgsm_labels, fgsm_preds)
plot_confusion(fgsm_labels, fgsm_preds, "MLP - FGSM Adversarial Test Data")

print("X_test_fgsm shape:", X_test_fgsm.shape)


In [ ]:
X_test_pgd, y_test_pgd, pgd_labels, pgd_preds = generate_adversarial_dataset(
    mlp,
    test_loader,
    attack_name="PGD",
    epsilon=PGD_EPSILON,
    alpha=PGD_ALPHA,
    steps=PGD_STEPS,
)

add_result(
    "MLP",
    "PGD",
    pgd_labels,
    pgd_preds,
    notes=f"epsilon={PGD_EPSILON}, alpha={PGD_ALPHA}, steps={PGD_STEPS}",
)
show_classification_summary("MLP", f"PGD epsilon={PGD_EPSILON}", pgd_labels, pgd_preds)
plot_confusion(pgd_labels, pgd_preds, "MLP - PGD Adversarial Test Data")

print("X_test_pgd shape:", X_test_pgd.shape)


## 11. Test Transferability to Classical Models

Here we reuse the FGSM and PGD examples generated against the MLP and evaluate the classical models on those same perturbed feature vectors.

If Logistic Regression or Random Forest performance drops, that suggests the adversarial examples are at least partially transferable across model types.


In [ ]:
adversarial_sets = {
    "FGSM": (X_test_fgsm, y_test_fgsm),
    "PGD": (X_test_pgd, y_test_pgd),
}

for model_name, model in sklearn_models.items():
    for attack_name, (X_adv, y_adv) in adversarial_sets.items():
        adv_preds = model.predict(X_adv)
        add_result(model_name, attack_name, y_adv, adv_preds, notes="transfer attack generated from MLP")
        show_classification_summary(model_name, attack_name, y_adv, adv_preds)


## 12. Results Summary

This table is the main output of the experiment. For the final report, the most important comparison is clean performance versus adversarial performance.


In [ ]:
results_df = pd.DataFrame(results)
metric_cols = ["accuracy", "weighted_f1", "macro_f1", "balanced_accuracy"]
results_display = results_df.copy()
for col in metric_cols:
    results_display[col] = results_display[col].map(lambda value: f"{value:.4f}")

display(results_display)

results_df.to_csv(OUTPUT_DIR / "results_summary.csv", index=False)
with open(OUTPUT_DIR / "results_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved results to {OUTPUT_DIR / 'results_summary.csv'}")
print(f"Saved results to {OUTPUT_DIR / 'results_summary.json'}")


In [ ]:
plt.figure(figsize=(10, 5))
plot_df = results_df.copy()
plot_df["label"] = plot_df["model"] + " - " + plot_df["condition"]
plt.bar(plot_df["label"], plot_df["accuracy"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Accuracy")
plt.title("Clean vs. Adversarial Accuracy by Model")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 13. Save Reproducibility Artifacts

This section saves the scaler, label encoder, model files, feature list, and experiment configuration. These are necessary if we want to reload the trained models later without rerunning the full notebook.


In [ ]:
# Scikit-learn artifacts
joblib.dump(scaler, MODEL_DIR / "scaler.joblib")
joblib.dump(label_encoder, MODEL_DIR / "label_encoder.joblib")
joblib.dump(log_reg, MODEL_DIR / "logistic_regression_model.joblib")
joblib.dump(random_forest, MODEL_DIR / "random_forest_model.joblib")

# PyTorch model state
mlp_artifact = {
    "state_dict": mlp.state_dict(),
    "input_dim": X_train_scaled.shape[1],
    "num_classes": len(class_names),
    "class_names": class_names,
    "dropout": DROPOUT,
}
torch.save(mlp_artifact, MODEL_DIR / "mlp_model.pth")

# Feature/config metadata
with open(OUTPUT_DIR / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

experiment_config = {
    "random_state": RANDOM_STATE,
    "data_metadata": data_metadata,
    "test_size": TEST_SIZE,
    "batch_size": BATCH_SIZE,
    "mlp_epochs": MLP_EPOCHS,
    "mlp_learning_rate": MLP_LEARNING_RATE,
    "dropout": DROPOUT,
    "sklearn_train_limit": SKLEARN_TRAIN_LIMIT,
    "fgsm_epsilon": FGSM_EPSILON,
    "pgd_epsilon": PGD_EPSILON,
    "pgd_alpha": PGD_ALPHA,
    "pgd_steps": PGD_STEPS,
    "class_names": class_names,
}
with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print(f"Saved artifacts under: {OUTPUT_DIR.resolve()}")


## 14. Results from the Original Notebook Runs

These are the key results from the two notebooks before cleanup. They are included as a historical reference so we do not lose the progress already made.

### Multiclass Kaggle run

| Model | Clean Accuracy | Clean Weighted F1 | FGSM Accuracy | FGSM Weighted F1 | PGD Accuracy | PGD Weighted F1 |
|---|---:|---:|---:|---:|---:|---:|
| MLP | 0.9860 | 0.9853 | 0.8708 | 0.8715 | 0.8508 | 0.8506 |
| Logistic Regression | 0.9741 | 0.9741 | 0.8574 | 0.8512 | 0.8463 | 0.8414 |
| Random Forest | 0.9982 | 0.9982 | 0.8404 | 0.7769 | 0.8455 | 0.7859 |

### Original preprocessing notebook run

| Model | Task | Accuracy | Notes |
|---|---|---:|---|
| Logistic Regression | Binary BENIGN vs ATTACK | 0.9749 | Strong baseline; recall for attack class was lower than Random Forest/MLP. |
| Random Forest | Binary BENIGN vs ATTACK | 0.9994 | Best clean binary result. |
| scikit-learn MLP | Binary BENIGN vs ATTACK | 0.9970 | Strong clean binary result, but not used for FGSM/PGD because PyTorch is easier for gradient-based attacks. |

Takeaway: the cleaned combined notebook should preserve Amogh's adversarial attack pipeline while keeping our stronger preprocessing, stratification, reporting, and artifact-saving structure.


## 15. Current Project Status and Next Steps

At this point, the project has a working experimental backbone:

- Clean-data intrusion detection baselines.
- A PyTorch neural-network IDS model.
- FGSM and PGD adversarial evaluation.
- Transferability testing against Logistic Regression and Random Forest.
- Saved results and model artifacts.

Recommended next steps after this cleanup:

1. Add an epsilon sweep to show how attack strength changes accuracy/F1.
2. Add stronger class-imbalance analysis, especially macro F1 and minority-class recall.
3. Add at least one defense, such as adversarial training.
4. Move reusable functions into `src/` after the notebook is stable.
